# [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CloudP0nd/gemma4-e2b-ja-lora-finetune/blob/main/Gemma4-E2B_JA_LoRA_FineTuning.ipynb)

# Gemma 4 E2B 日本語 LoRA ファインチューニング（破滅的忘却対策版）

## 📋 概要

- **対象モデル**: `unsloth/gemma-4-E2B-it` (4bit 量子化)
- **フレームワーク**: Unsloth (`FastModel` API)
- **データセット**: `llm-jp/oasst1-21k-ja` から抽出した高品質 8,000 件
- **手法**: QLoRA（4bit NF4 量子化 + LoRA アダプタ）
- **目的**: 日本語性能を底上げしつつ、破滅的忘却（catastrophic forgetting）を回避

## 🆕 Gemma 4 について

Gemma 4 は Google DeepMind がリリースした新しいオープンモデルファミリーで、E2B / E4B / 12B / 26B-A4B / 31B の5サイズ展開、**Text/Image/Audio のマルチモーダル**対応、最大 256K コンテキストをサポートします（Apache-2.0）。E2B は Dense + PLE 構造で、128K コンテキストまで対応、エッジ推論向けの軽量モデルです。

> ⚠️ Gemma 4 は `FastLanguageModel` ではなく **`FastModel`** API を使用します（マルチモーダル対応のため）。テキストのみ学習する場合でも、`finetune_vision_layers=False` を指定して視覚層をフリーズします。

## 🛡️ 破滅的忘却への対策

| # | 施策 | 本ノートブックでの設定 |
|---|------|----------------------|
| 1 | QLoRA（4bit 量子化 + LoRA）| rank=16 / alpha=32。ベース重みを4bitで凍結し、アダプタのみ学習 |
| 2 | attention + MLP 両方へ適用 | `finetune_attention_modules=True`, `finetune_mlp_modules=True` |
| 3 | 視覚層はフリーズ | `finetune_vision_layers=False` |
| 4 | 学習率を高すぎない | `2e-4`（LoRA 標準的） |
| 5 | エポックを絞る | 2 エポック |
| 6 | Cosine LR Schedule | 学習終盤で緩やかに減衰 |
| 7 | Warmup (5%) | 初期の不安定更新を抑制 |
| 8 | Weight Decay (0.01) | L2 正則化 |
| 9 | LoRA Dropout (0.05) | アダプタ過適合防止 |
| 10 | 学習前後の Perplexity 評価 | 忘却の定量モニタリング |
| 11 | `train_on_responses_only` | 応答部分のみ学習（プロンプト部はマスク） |

## ⚙️ ハイパーパラメータ

| 項目 | 値 |
|---|---|
| LoRA Rank | 16 |
| LoRA Alpha | 32 |
| LoRA Dropout | 0.05 |
| 学習対象 | language + attention + MLP（視覚はフリーズ） |
| Gradient Accumulation Steps | 4 |
| Learning Rate | 2e-4 |
| Optimizer | `paged_adamw_8bit` |
| LR Scheduler | Cosine |
| Epochs | 2 |
| Max Sequence Length | 2048 |
| Precision | 4bit NF4 量子化（`load_in_4bit=True`） |

## 🚀 実行環境

- Google Colab Pro 推奨（A100 / L4）
- T4 (16GB) でも動作可能
- 推論時推奨パラメータ: `temperature=1.0, top_p=0.95, top_k=64`（Gemma 4 公式推奨）


## Step 0: GPU 確認


In [ ]:
!nvidia-smi


## Step 1: ライブラリのインストール

Unsloth 公式の Gemma 4 レシピに従い、依存パッケージを Colab 環境に合わせてインストールします。


In [ ]:
%%capture
import os
# ★ フラグメンテーション解消: torch import 前に設定する必要がある
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps "transformers>=5.5.0,<6.0.0" "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec

# timm は Gemma 4 の vision/audio 用
!pip install --no-deps --upgrade timm

import torch
torch._dynamo.config.recompile_limit = 64
# ★ dynamo エラーを抑制（fallback 時の出力を減らす）
torch._dynamo.config.suppress_errors = True
print("✅ Installation complete")


## Step 2: HuggingFace ログイン

`unsloth/gemma-4-E2B-it` は gated model のため、HuggingFace でライセンス同意済みのトークンが必要です。
[https://huggingface.co/unsloth/gemma-4-E2B-it](https://huggingface.co/unsloth/gemma-4-E2B-it) で事前に同意してください。


In [ ]:
from huggingface_hub import login
import os

# Colab Secrets を優先、次に環境変数をフォールバック
HF_TOKEN = None

# 1. Colab Secrets から取得を試行
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    if HF_TOKEN:
        print("✅ HF_TOKEN loaded from Colab Secrets")
    else:
        HF_TOKEN = None
except ImportError:
    pass  # Colab 環境ではない
except ValueError:
    pass  # Secret が未設定、または空

# 2. 環境変数フォールバック
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    if HF_TOKEN:
        print("✅ HF_TOKEN loaded from environment variable")
    else:
        print("⚠️ HF_TOKEN が未設定です。Colab Secrets に登録するか、login() を手動で呼んでください。")

# 3. HuggingFace へログイン
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    login()  # インタラクティブにプロンプト表示


## Step 3: インポート


In [ ]:
import os
import re
import random
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset, Dataset
from trl import SFTTrainer, SFTConfig
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, standardize_data_formats, train_on_responses_only

sns.set_style("whitegrid")
random.seed(3407)
np.random.seed(3407)
torch.manual_seed(3407)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Step 4: モデル読み込み（4bit 量子化 / QLoRA 用）

- **`dtype=None`**: Unsloth 自動検出
- **`load_in_4bit=True`**: ★ QLoRA。ベース重みを 4bit NF4 で読み込み、VRAM を大幅に削減
- **`full_finetuning=False`**: LoRA モード（ベースは凍結、アダプタのみ学習）
- **`max_seq_length=2048`**: 指定通り
- **効果**: T4 (16GB) でも余裕で動作、学習速度もほぼ FP16 と同等


In [ ]:
MAX_SEQ_LENGTH = 2048
MODEL_NAME = "unsloth/gemma-4-E2B-it"

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    dtype=None,                # 自動検出
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,         # ★ QLoRA: 4bit NF4 量子化でベースを読み込み
    full_finetuning=False,     # ★ LoRA モード（ベースは凍結）
    text_only=True,            # ★ ビジョン/オーディオ塔をスキップ（VRAM 節約・T4 で安定）
    fullgraph=False,           # ★ graph break を許可（fused CE loss の torch.func.grad が
                               #   dynamo で tracing できない問題を回避。eager fallback を防ぐ）
)


## Step 4.5: Unsloth Gemma4 パッチの use_cache バグ修正

Unsloth の Gemma4 KV共有パッチ (`unsloth_zoo.temporary_patches.gemma4`) は、
`use_cache=False` 時に内部で `DynamicCache` を注入しますが、その呼び出し方が
`transformers` の `@merge_with_config_defaults` デコレータと衝突し、
`got multiple values for argument 'use_cache'` エラーを引き起こします。

本セルはパッチを安全な実装に置き換え、PPL 評価・学習の両方で
`use_cache=False` が安全に使えるようにします。


In [ ]:
# ============================================================
# ⚠ Unsloth Gemma4 パッチの use_cache 重複バグを修正
# ============================================================
# unsloth_zoo.temporary_patches.gemma4 の KV共有パッチは、
# use_cache=False/None のとき DynamicCache を注入して use_cache=True
# に書き換えるが、その際 _orig_forward(*inner_args, **inner_kwargs) と呼び出し、
# inner_args 側に use_cache が位置引数として並ぶ。
# 一方 transformers の @merge_with_config_defaults デコレータは
# 位置引数の use_cache を見つけると kwargs にも use_cache を追加してしまい、
# "got multiple values for argument 'use_cache'" エラーになる。
#
# 本セルはパッチを安全な実装に置き換える:
# - 位置引数で渡された use_cache/past_key_values を kwargs に正規化
# - DynamicCache 注入後は kwargs のみで _orig_forward を呼ぶ
#
# これにより PPL 評価・学習の両方で use_cache=False が安全に使える。
# (E2B/E4B で KV共有 + use_cache=False のゴミ出力問題もパッチ本体が解決)
import inspect
from transformers.models.gemma4.modeling_gemma4 import Gemma4TextModel

_patched_forward = Gemma4TextModel.forward

# closure から _orig_forward を抽出
_orig_forward = None
for _cell in (_patched_forward.__closure__ or []):
    _v = _cell.cell_contents
    if callable(_v) and not inspect.isclass(_v):
        _orig_forward = _v
        break

if _orig_forward is None:
    print("⚠ Could not extract _orig_forward from unsloth patch. Skipping monkey-fix.")
else:
    try:
        _sig = inspect.signature(_orig_forward)
    except (TypeError, ValueError):
        _sig = None

    def _fixed_forward(self, *args, **kwargs):
        num_kv_shared = getattr(getattr(self, "config", None), "num_kv_shared_layers", 0)
        if not num_kv_shared or num_kv_shared <= 0:
            return _orig_forward(self, *args, **kwargs)

        # Resolve past_key_values / use_cache (位置・キーワード両対応)
        if _sig is not None:
            try:
                bound = _sig.bind_partial(self, *args, **kwargs)
                arguments = bound.arguments
                past_key_values = arguments.get("past_key_values", None)
                use_cache_kw = arguments.get("use_cache", None)
            except TypeError:
                return _orig_forward(self, *args, **kwargs)
        else:
            past_key_values = kwargs.get("past_key_values", None)
            use_cache_kw = kwargs.get("use_cache", None)

        use_cache_resolved = (
            use_cache_kw
            if use_cache_kw is not None
            else getattr(self.config, "use_cache", True)
        )

        # use_cache=True or past_key_values 指定時は素通し
        if past_key_values is not None or use_cache_resolved:
            return _orig_forward(self, *args, **kwargs)

        # use_cache=False/None + past_key_values=None → DynamicCache を注入
        # ⚠ 安全な呼び出し方: 位置引数で渡された param があれば kwargs に正規化してから
        # _orig_forward(self, *args, **kwargs) を呼ぶ。
        # こうすると @merge_with_config_defaults が use_cache を kwargs に追加しても
        # 重複しない (kwargs 側に上書きされるだけ)。
        from transformers.cache_utils import DynamicCache
        kwargs["past_key_values"] = DynamicCache(config=self.config)
        kwargs["use_cache"] = True
        # 位置引数で use_cache / past_key_values が渡されていた場合は args 側を
        # クリーンアップする必要があるが、Unsloth の compiled forward は
        # 全て kwargs 渡しなのでここは素通しで安全。
        return _orig_forward(self, *args, **kwargs)

    _fixed_forward.__name__ = "_fixed_gemma4_forward"
    _fixed_forward.__qualname__ = "Gemma4TextModel.forward"
    if hasattr(_patched_forward, "__doc__"):
        _fixed_forward.__doc__ = _patched_forward.__doc__
    if hasattr(_patched_forward, "__wrapped__"):
        _fixed_forward.__wrapped__ = _patched_forward.__wrapped__

    Gemma4TextModel.forward = _fixed_forward
    print("✅ Patched Gemma4TextModel.forward to fix use_cache duplicate bug")
    print(f"   (num_kv_shared_layers = {getattr(Gemma4TextModel, '_num_kv_shared_layers', 'unknown')})")


## Step 5: LoRA 適用

Gemma 4 は `FastModel.get_peft_model` を使います。`target_modules` リストではなく、
**`finetune_*_modules` フラグ**で学習対象を制御します。

ユーザー指定の「attention 全層 + MLP 全層」は以下で等価に実現できます：

- `finetune_vision_layers=False` → 視覚層フリーズ（テキストのみ学習）
- `finetune_language_layers=True` → 言語層は学習
- `finetune_attention_modules=True` → q,k,v,o_proj 系すべて学習
- `finetune_mlp_modules=True` → gate,up,down_proj 系すべて学習


In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # テキストのみ学習（視覚はフリーズ）
    finetune_language_layers   = True,   # 言語層は学習
    finetune_attention_modules = True,   # ★ attention 全層（q,k,v,o_proj 系）
    finetune_mlp_modules       = True,   # ★ MLP 全層（gate,up,down_proj 系）

    r = 16,                # ★ LoRA Rank
    lora_alpha = 32,       # ★ LoRA Alpha (2 * r)
    lora_dropout = 0.05,   # ★ 忘却防止にドロップアウト
    bias = "none",
    random_state = 3407,
)

# 学習可能パラメータ数を確認
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} ({100 * trainable / total:.3f}% of {total:,})")


## Step 6: チャットテンプレートの設定

Gemma 4 は専用のチャットテンプレート `"gemma-4"` を使います。
`<|turn>user\n` / `<|turn>model\n` 形式のロールマーカーを使用します。


In [ ]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-4",
)
print("✅ Gemma 4 chat template applied")


## Step 7: データセット読み込み

`llm-jp/oasst1-21k-ja` を読み込みます。


In [ ]:
raw_dataset = load_dataset("llm-jp/oasst1-21k-ja", split="train")
print(f"Total samples in dataset: {len(raw_dataset):,}")
print(f"Columns: {raw_dataset.column_names}")
print(f"\nSample example:\n{raw_dataset[0]}")


## Step 8: データ正規化と高品質サンプル抽出（8,000 件）

1. `standardize_data_formats` で `conversations` 形式に統一
2. 日本語含有率・長さ・重複でフィルタ → 8,000 件抽出
3. `formatting_prompts_func` で `text` フィールドにフォーマット


In [ ]:
# まず standardize して conversations 形式に統一
std_dataset = standardize_data_formats(raw_dataset)
print(f"After standardize: {len(std_dataset):,} samples")
print(f"Columns: {std_dataset.column_names}")
print(f"\nSample:\n{std_dataset[0]}")


In [ ]:
JAPANESE_PATTERN = re.compile(r"[\u3040-\u309F\u30A0-\u30FF\u4E00-\u9FFF]")
URL_ONLY_PATTERN = re.compile(r"^(https?://\S+\s*)+$")


def japanese_ratio(text: str) -> float:
    if not text:
        return 0.0
    jp_chars = len(JAPANESE_PATTERN.findall(text))
    return jp_chars / max(len(text), 1)


def extract_text_from_conversations(convos: list) -> tuple:
    """conversations から (instruction, output) を抽出"""
    instruction = ""
    output = ""
    for c in convos:
        role = c.get("role", "").lower()
        content = c.get("content", "")
        if role in ("human", "user") and not instruction:
            instruction = content
        elif role in ("gpt", "assistant", "model") and not output:
            output = content
    return instruction, output


def is_high_quality(example: dict) -> bool:
    convos = example.get("conversations", [])
    if not convos:
        return False
    instruction, output = extract_text_from_conversations(convos)
    if not instruction or not output:
        return False
    if len(output) < 50 or len(output) > 1500:
        return False
    if len(instruction) < 5 or len(instruction) > 1000:
        return False
    if japanese_ratio(output) < 0.30:
        return False
    if japanese_ratio(instruction) < 0.10:
        return False
    if URL_ONLY_PATTERN.match(output.strip()):
        return False
    return True


# フィルタリング
print("Filtering high-quality Japanese samples...")
filtered_records = []
seen_instructions = set()

for example in std_dataset:
    if not is_high_quality(example):
        continue
    instr, _ = extract_text_from_conversations(example["conversations"])
    if instr in seen_instructions:
        continue
    seen_instructions.add(instr)
    filtered_records.append(example)

print(f"After filtering: {len(filtered_records):,} samples")

# シャッフルして 8,000 件
random.shuffle(filtered_records)
N_SAMPLES = 8000
if len(filtered_records) < N_SAMPLES:
    print(f"⚠️ {N_SAMPLES} 件に満たないため全件使用: {len(filtered_records)}")
    selected = filtered_records
else:
    selected = filtered_records[:N_SAMPLES]

# Dataset 化
filtered_dataset = Dataset.from_list(selected)
print(f"Selected samples: {len(filtered_dataset):,}")


## Step 9: チャットテンプレートでフォーマット

`tokenizer.apply_chat_template(conversations, ...)` を通して `text` フィールドを作ります。
Unsloth 公式レシピ通り、先頭の `<bos>` は除去します。


In [ ]:
def formatting_prompts_func(examples):
    convos_list = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        ).removeprefix("<bos>")
        for convo in convos_list
    ]
    return {"text": texts}


filtered_dataset = filtered_dataset.map(formatting_prompts_func, batched=True)
print(f"Formatted dataset: {len(filtered_dataset):,} samples")
print(f"\n=== Sample text (first 600 chars) ===")
print(filtered_dataset[0]["text"][:600])


## Step 10: 学習前評価（ベースライン Perplexity）

破滅的忘却を定量モニタリングするため、**学習前の Perplexity（日本語 / 英語）** を計測します。


In [ ]:
# VRAM に応じて per_device_batch と seq_length を調整
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
PER_DEVICE_BATCH = 2 if total_vram_gb > 20 else 1

# ★ T4 (≤16GB) では seq_length を下げて OOM 回避
#   Gemma4 は vocab=256k と大きく、logits 計算が (batch, seq, 256k) × 4byte(float32) で
#   seq=2048 だと 2GB、.float().contiguous() で更に 2GB 必要 → T4 では厳しい
#   seq=1024 なら 1GB + 1GB = 2GB で余裕
if total_vram_gb <= 16:
    EFFECTIVE_SEQ_LENGTH = 1024
    GRAD_ACCUM = 8  # 実効バッチサイズ 8 を維持 (1 × 8)
    print(f"⚠️ Low VRAM ({total_vram_gb:.1f}GB) detected.")
    print(f"   Reducing max_seq_length: {MAX_SEQ_LENGTH} -> {EFFECTIVE_SEQ_LENGTH}")
    print(f"   Increasing gradient_accumulation_steps: 4 -> {GRAD_ACCUM} (実効バッチサイズ 8 を維持)")
    print(f"   ※ 長いサンプルはトランケートされます（出力末尾がカット）")
else:
    EFFECTIVE_SEQ_LENGTH = MAX_SEQ_LENGTH
    GRAD_ACCUM = 4

sft_config = SFTConfig(
    output_dir="./outputs",
    dataset_text_field="text",
    num_train_epochs=2,                       # ★ エポック数
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,   # ★ VRAM に応じて 4 or 8
    learning_rate=2e-4,                       # ★ 指定値
    optim="paged_adamw_8bit",                 # ★ 指定値
    lr_scheduler_type="cosine",               # ★ 忘却防止: cosine
    warmup_ratio=0.05,                        # 5% warmup
    weight_decay=0.01,                        # L2 正則化
    max_grad_norm=1.0,
    # fp16 / bf16 は Unsloth 側で自動処理（4bit QLoRA 時は明示不要）
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    seed=3407,
    report_to="none",
    dataloader_num_workers=2,
    max_seq_length=EFFECTIVE_SEQ_LENGTH,      # ★ VRAM に応じて 1024 or 2048
    packing=False,                            # ★ 文境界を保持（train_on_responses_only との整合）
    dataset_num_proc=2,                       # ★ トークナイズ並列化で前処理高速化
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=filtered_dataset,
    eval_dataset=None,
    args=sft_config,
)

print(f"\nEffective batch size: {PER_DEVICE_BATCH} * {GRAD_ACCUM} = {PER_DEVICE_BATCH * GRAD_ACCUM}")
print(f"Max seq length: {EFFECTIVE_SEQ_LENGTH}")
print(f"Total samples: {len(filtered_dataset):,}")
print(f"Steps per epoch: {len(filtered_dataset) // (PER_DEVICE_BATCH * GRAD_ACCUM):,}")
print(f"Total steps: {2 * len(filtered_dataset) // (PER_DEVICE_BATCH * GRAD_ACCUM):,}")


## Step 11: 学習設定

指定ハイパーパラメータで `SFTTrainer`（`SFTConfig` 使用）を構築します。


In [ ]:
# VRAM に応じて per_device_batch を調整
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
PER_DEVICE_BATCH = 2 if total_vram_gb > 20 else 1

sft_config = SFTConfig(
    output_dir="./outputs",
    dataset_text_field="text",
    num_train_epochs=2,                       # ★ エポック数
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=4,            # ★ 指定値
    learning_rate=2e-4,                       # ★ 指定値
    optim="paged_adamw_8bit",                 # ★ 指定値
    lr_scheduler_type="cosine",               # ★ 忘却防止: cosine
    warmup_ratio=0.05,                        # 5% warmup
    weight_decay=0.01,                        # L2 正則化
    max_grad_norm=1.0,
    # fp16 / bf16 は Unsloth 側で自動処理（4bit QLoRA 時は明示不要）
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    seed=3407,
    report_to="none",
    dataloader_num_workers=2,
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,                            # ★ 文境界を保持（train_on_responses_only との整合）
    dataset_num_proc=2,                       # ★ トークナイズ並列化で前処理高速化
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=filtered_dataset,
    eval_dataset=None,
    args=sft_config,
)

print(f"Effective batch size: {PER_DEVICE_BATCH} * 4 = {PER_DEVICE_BATCH * 4}")
print(f"Total samples: {len(filtered_dataset):,}")
print(f"Steps per epoch: {len(filtered_dataset) // (PER_DEVICE_BATCH * 4):,}")
print(f"Total steps: {2 * len(filtered_dataset) // (PER_DEVICE_BATCH * 4):,}")


## Step 12: 応答部分のみ学習（`train_on_responses_only`）

ユーザー発言部分をマスクし、モデル応答部分のみで loss を計算するようにします。
Gemma 4 のロールマーカーは `<|turn>user\n` と `<|turn>model\n` です。


In [ ]:
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)

# ★ マスクの厳密な検証
sample = trainer.train_dataset[0]
labels = sample["labels"]
n_total = len(labels)
n_learned = int((np.array(labels) != -100).sum())
print(f"Sample 0: total={n_total} tokens, learned={n_learned} tokens ({100 * n_learned / max(n_total, 1):.1f}%)")

if n_learned == 0:
    raise RuntimeError(
        "❌ train_on_responses_only が応答部分を検出していません。\n"
        "マーカー '<|turn>user\\n' / '<|turn>model\\n' が実際のテンプレートと一致しているか確認してください。"
    )

print("\n=== Full input (decoded, first 500 chars) ===")
print(tokenizer.decode(sample["input_ids"])[:500])
print("\n=== Learned tokens only (first 500 chars) ===")
learned_ids = [t for t, l in zip(sample["input_ids"], labels) if l != -100]
print(tokenizer.decode(learned_ids)[:500])


## Step 13: 学習前メモリ状態


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")


## Step 14: 学習実行


In [ ]:
import time
start = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start
print(f"\nTraining completed in {elapsed/60:.1f} minutes")
print(f"Final loss: {trainer_stats.training_loss:.4f}")


## Step 15: 学習後メモリ統計


In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max = {used_percentage} %.")
print(f"Peak reserved memory for training % of max = {lora_percentage} %.")


## Step 16: 学習曲線の可視化


In [ ]:
log_history = trainer.state.log_history
losses = [entry["loss"] for entry in log_history if "loss" in entry]
steps = [entry["step"] for entry in log_history if "loss" in entry]

plt.figure(figsize=(10, 4))
plt.plot(steps, losses, alpha=0.6, label="loss")
if len(losses) > 10:
    window = 10
    ma = np.convolve(losses, np.ones(window)/window, mode="valid")
    plt.plot(steps[window-1:], ma, color="red", linewidth=2, label=f"MA({window})")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title(f"Training Loss Curve (final: {losses[-1]:.4f})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Step 17: 学習後評価（Perplexity 比較）

日本語 PPL が下がり、英語 PPL が維持されていれば、破滅的忘却なく日本語能力が向上した証拠です。


In [ ]:
print("Computing post-training PPL...")
ppl_ja_after = compute_ppl(model, tokenizer, EVAL_JA_TEXTS[:20])
ppl_en_after = compute_ppl(model, tokenizer, EVAL_EN_TEXTS[:20])

print(f"\n=== PPL Comparison ===")
print(f"{'Metric':<15} {'Before':>10} {'After':>10} {'Δ':>10}")
print(f"{'-'*50}")
print(f"{'Japanese PPL':<15} {ppl_ja_before:>10.3f} {ppl_ja_after:>10.3f} {ppl_ja_after - ppl_ja_before:>+10.3f}")
print(f"{'English  PPL':<15} {ppl_en_before:>10.3f} {ppl_en_after:>10.3f} {ppl_en_after - ppl_en_before:>+10.3f}")
print()
if ppl_ja_after < ppl_ja_before:
    print("✅ 日本語 PPL が改善しました")
else:
    print("⚠️ 日本語 PPL が上昇しました")
if ppl_en_after - ppl_en_before < 1.0:
    print("✅ 英語 PPL は概ね維持されています（破滅的忘却なし）")
else:
    print("⚠️ 英語 PPL が上昇しました（破滅的忘却の可能性あり）")


## Step 18: 生成サンプル比較

Gemma 4 公式推奨パラメータ `temperature=1.0, top_p=0.95, top_k=64` を使用します。


In [ ]:
from transformers import TextStreamer
FastModel.for_inference(model)  # 推論モード

# ★ generate 用に left padding を明示（right padding だと生成が崩れる）
tokenizer.padding_side = "left"


def generate_response(prompt: str, max_new_tokens: int = 256) -> str:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=1.0,    # ★ Gemma 4 推奨
            top_p=0.95,         # ★ Gemma 4 推奨
            top_k=64,           # ★ Gemma 4 推奨
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return response


test_prompts = [
    "日本の四季について説明してください。",
    "Python でリストを逆順にする方法を教えてください。",
    "健康的な生活を送るための 3 つのアドバイスをお願いします。",
    "What is the difference between AI and machine learning?",
]

for prompt in test_prompts:
    print("=" * 70)
    print(f"USER: {prompt}")
    print(f"MODEL: {generate_response(prompt)}")
    print()


## Step 19: モデル保存

### 19-1: LoRA アダプタのみ保存（軽量・推奨）


In [ ]:
ADAPTER_DIR = "./gemma4-e2b-ja-lora"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"✅ LoRA adapter saved to {ADAPTER_DIR}")
!ls -lh {ADAPTER_DIR}


### 19-2: マージ済み 16bit モデルとして保存

HuggingFace Hub にアップロードする場合はこちらを推奨。


In [ ]:
# MERGED_DIR = "./gemma4-e2b-ja-merged"
# model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
# print(f"✅ Merged 16bit model saved to {MERGED_DIR}")


### 19-3: HuggingFace Hub へアップロード


In [ ]:
# HF_REPO_ID = "your-username/gemma4-e2b-ja-lora"  # ★ 書き換えてください
# # login() 済みなら token 引数は省略可能（Colab Secrets / 環境変数の HF_TOKEN が自動使用される）
# model.push_to_hub(HF_REPO_ID)
# tokenizer.push_to_hub(HF_REPO_ID)


## Step 20: （オプション）保存済みアダプタの再読込テスト

学習したアダプタを新規セッションで読み込む際のコード例。


In [ ]:
# if False:
#     from unsloth import FastModel
#     model, tokenizer = FastModel.from_pretrained(
#         model_name=ADAPTER_DIR,   # 保存したアダプタ名
#         max_seq_length=MAX_SEQ_LENGTH,
#         load_in_4bit=True,        # 推論時は 4bit でOK
#     )


## Step 21: サマリ


In [ ]:
print("=" * 70)
print("Training Summary")
print("=" * 70)
print(f"Model:          {MODEL_NAME}")
print(f"Dataset:        llm-jp/oasst1-21k-ja (filtered to {len(filtered_dataset):,} samples)")
print(f"LoRA rank:      16 / alpha: 32 / dropout: 0.05")
print(f"Quantization:   4bit NF4 (QLoRA)")
print(f"Targets:        language + attention + MLP (vision frozen)")
print(f"LR:             2e-4 / optim: paged_adamw_8bit / scheduler: cosine")
print(f"Epochs:         2")
print(f"Max seq len:    {MAX_SEQ_LENGTH}")
print(f"Training time:  {elapsed/60:.1f} min")
print(f"Final loss:     {trainer_stats.training_loss:.4f}")
print()
print(f"PPL (JA): {ppl_ja_before:.3f} -> {ppl_ja_after:.3f}")
print(f"PPL (EN): {ppl_en_before:.3f} -> {ppl_en_after:.3f}")
print("=" * 70)
